Ny variant som hämtar alla positioner för båtar
* denna Notebook [434_2_samtrafiken-vechicle-id.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/blob/main/Notebook/434_2_samtrafiken-vechicle-id.ipynb)
* se även [434_samtrafiken-vechicle-id.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/blob/main/Notebook/434_samtrafiken-vechicle-id.ipynb)
* Issue [#434](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/434)

In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-09-12 16:41


In [2]:
%pip install gtfs-realtime-bindings 

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import requests
import zipfile
import io
import html
import pandas as pd
import folium

from google.transit import gtfs_realtime_pb2

API_KEY = os.environ["GTFSSweden3"]
STATIC_API_KEY = os.environ["GTFSSweden3Static"]

print("API keys loaded")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.1) doesn't match a supported version!
  warnings.warn(


API keys loaded


In [4]:
URL = (
    "https://opendata.samtrafiken.se/gtfs-rt-sweden/sl/"
    "VehiclePositionsSweden.pb"
    f"?key={API_KEY}"
)

print(URL.replace(API_KEY, "***"))

r = requests.get(URL)
print("HTTP:", r.status_code)

r.raise_for_status()

feed = gtfs_realtime_pb2.FeedMessage()
feed.ParseFromString(r.content)

print("Entities:", len(feed.entity))

https://opendata.samtrafiken.se/gtfs-rt-sweden/sl/VehiclePositionsSweden.pb?key=***
HTTP: 200
Entities: 928


In [5]:
rows = []

for entity in feed.entity:
    if not entity.HasField("vehicle"):
        continue

    v = entity.vehicle

    rows.append({
        "vehicle_id": v.vehicle.id,
        "trip_id": v.trip.trip_id,
        "route_id": v.trip.route_id,
        "lat": v.position.latitude,
        "lon": v.position.longitude,
        "timestamp": v.timestamp,
    })

live = pd.DataFrame(rows)

print("Antal fordon:", len(live))
display(live.head())

Antal fordon: 928


,vehicle_id,trip_id,route_id,lat,lon,timestamp
0,9031001001004806,14010000721661683,,59.331226,18.023401,1789224112
1,9031001003005037,14010000731937090,,59.166775,18.173222,1789224113
2,9031001003005052,14010000731937617,,59.168400,18.162279,1789224112
3,9031001003005363,14010000704104719,,59.330051,18.401350,1789224112
4,9031001007500531,14010000721404910,,59.323978,18.094273,1789224113


In [6]:
import io
import zipfile
import pandas as pd
import requests

STATIC_URL = (
    "https://opendata.samtrafiken.se/gtfs-sweden/sweden.zip"
    f"?key={STATIC_API_KEY}"
)

r_static = requests.get(STATIC_URL)
r_static.raise_for_status()

z = zipfile.ZipFile(io.BytesIO(r_static.content))

routes = pd.read_csv(
    z.open("routes.txt"),
    dtype=str
)

print("Antal routes:", len(routes))
display(routes.head())

Antal routes: 7323


,route_id,agency_id,route_short_name,route_long_name,route_type,route_desc
0,9011001000000000,505000000000000001,40,SL,106,NaN
1,9011001000000000-41,505000000000000001,41,SL,106,NaN
2,9011001000000000-43,505000000000000001,43,SL,106,NaN
3,9011001000000000-48,505000000000000001,48,SL,106,NaN
4,9011001000100000,505000000000000001,1,NaN,700,NaN


In [7]:
live_routes = live.merge(
    routes[["route_id", "route_short_name", "route_type"]],
    on="route_id",
    how="left"
)

print("Antal fordon:", len(live_routes))
print("\nRoute types:")
display(live_routes["route_type"].value_counts(dropna=False))

Antal fordon: 928

Route types:


route_type
NaN    926
700      2
Name: count, dtype: int64

In [8]:
print("Exempel från realtime:")
display(live[["vehicle_id", "trip_id", "route_id"]].head(10))

print("\nExempel på route_id från static:")
display(routes[["route_id", "route_short_name", "route_type"]].head(10))

Exempel från realtime:


,vehicle_id,trip_id,route_id
0,9031001001004806,14010000721661683,
1,9031001003005037,14010000731937090,
2,9031001003005052,14010000731937617,
3,9031001003005363,14010000704104719,
4,9031001007500531,14010000721404910,
5,9031001001007183,14010000702300013,
6,9031001003005509,14010000703792104,
7,9031001003002765,14010000676790148,
8,9031001004500419,14010000707553192,
9,9031001003005330,14010000677072059,



Exempel på route_id från static:


,route_id,route_short_name,route_type
0,9011001000000000,40,106
1,9011001000000000-41,41,106
2,9011001000000000-43,43,106
3,9011001000000000-48,48,106
4,9011001000100000,1,700
5,9011001000200000,2,700
6,9011001000300000,3,700
7,9011001000400000,4,700
8,9011001000500000,5,700
9,9011001000600000,6,700


In [9]:
trips = pd.read_csv(
    z.open("trips.txt"),
    dtype=str
)

print("Antal trips:", len(trips))
display(trips.head())

Antal trips: 368048


,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,shape_id,samtrafiken_internal_trip_number
0,9011001000000000,1,400000000000005073,NaN,NaN,0,4400000000000005073,2200
1,9011001000000000,2,400000000000005074,NaN,NaN,0,4400000000000005074,2200
2,9011001000000000,3,400000000000005077,NaN,NaN,0,4400000000000005077,2200
3,9011001000000000,4,400000000000005079,NaN,NaN,0,4400000000000005079,2200
4,9011001000000000,5,400000000000005080,NaN,NaN,0,4400000000000005080,2200


In [10]:
live_trips = live.merge(
    trips[[
        "trip_id",
        "route_id"
    ]],
    on="trip_id",
    how="left",
    suffixes=("", "_static")
)

print("Antal fordon:", len(live_trips))
print("Trip-id matchade:", live_trips["route_id_static"].notna().sum())
print("Trip-id saknas:", live_trips["route_id_static"].isna().sum())

display(
    live_trips[
        ["vehicle_id", "trip_id", "route_id", "route_id_static", "lat", "lon"]
    ].head(10)
)

Antal fordon: 928
Trip-id matchade: 884
Trip-id saknas: 44


,vehicle_id,trip_id,route_id,route_id_static,lat,lon
0,9031001001004806,14010000721661683,,9011001005400000,59.331226,18.023401
1,9031001003005037,14010000731937090,,9011001080700000,59.166775,18.173222
2,9031001003005052,14010000731937617,,9011001080700000,59.168400,18.162279
3,9031001003005363,14010000704104719,,9011001042200000,59.330051,18.401350
4,9031001007500531,14010000721404910,,9011001008200000,59.323978,18.094273
5,9031001001007183,14010000702300013,,9011001020100000,59.357590,18.102352
6,9031001003005509,14010000703792104,,9011001018100000,59.258453,18.106171
7,9031001003002765,14010000676790148,,9011001072500000,59.198891,17.764021
8,9031001004500419,14010000707553192,,9011001056800000,59.527172,17.915480
9,9031001003005330,14010000677072059,,9011001044300000,59.314140,18.101915


In [11]:
live_routes = live_trips.merge(
    routes[
        ["route_id", "route_short_name", "route_type"]
    ],
    left_on="route_id_static",
    right_on="route_id",
    how="left"
)

print("Antal fordon:", len(live_routes))

display(
    live_routes[
        [
            "vehicle_id",
            "trip_id",
            "route_id_static",
            "route_short_name",
            "route_type",
            "lat",
            "lon"
        ]
    ].head(20)
)

Antal fordon: 928


,vehicle_id,trip_id,route_id_static,route_short_name,route_type,lat,lon
0,9031001001004806,14010000721661683,9011001005400000,54,700,59.331226,18.023401
1,9031001003005037,14010000731937090,9011001080700000,807,700,59.166775,18.173222
2,9031001003005052,14010000731937617,9011001080700000,807,700,59.168400,18.162279
3,9031001003005363,14010000704104719,9011001042200000,422,700,59.330051,18.401350
4,9031001007500531,14010000721404910,9011001008200000,82,1000,59.323978,18.094273
5,9031001001007183,14010000702300013,9011001020100000,201,700,59.357590,18.102352
6,9031001003005509,14010000703792104,9011001018100000,181,700,59.258453,18.106171
7,9031001003002765,14010000676790148,9011001072500000,725,700,59.198891,17.764021
8,9031001004500419,14010000707553192,9011001056800000,568,700,59.527172,17.915480
9,9031001003005330,14010000677072059,9011001044300000,443,700,59.314140,18.101915


In [12]:
boats = live_routes[live_routes["route_type"] == "1000"].copy()

In [13]:
boats = live_routes[
    live_routes["route_type"] == "1000"
].copy()

print("Antal båtar:", len(boats))

display(
    boats[
        [
            "vehicle_id",
            "trip_id",
            "route_id_static",
            "route_short_name",
            "route_type",
            "lat",
            "lon"
        ]
    ].sort_values("route_short_name")
)

Antal båtar: 37


,vehicle_id,trip_id,route_id_static,route_short_name,route_type,lat,lon
67,9031001000500502,14010000725964743,9011114001100000,11,1000,59.375134,18.336308
157,9031001000500547,14010000729757553,9011114001100000,11,1000,59.338928,18.209742
383,9031001000500679,14010000733933240,9011114001200000,12,1000,59.457111,18.649292
841,9031001000500536,14010000729759138,9011114001200000,12,1000,59.462147,18.631741
471,9031001000500539,14010000729761729,9011114001300000,13,1000,59.415150,18.503790
329,9031001000500537,14010000729760454,9011114001300000,13,1000,59.416988,18.353119
789,9031001000500503,14010000725967704,9011114001400000,14,1000,59.401001,18.572548
348,9031001000500535,14010000732572657,9011114001400000,14,1000,59.378780,18.871216
223,9031001000500548,14010000729766531,9011114001600000,16,1000,59.303001,18.781807
903,9031001000500540,14010000719263554,9011114001600000,16,1000,59.298382,18.765450


In [14]:
boats

,vehicle_id,trip_id,route_id_x,lat,lon,timestamp,route_id_static,route_id_y,route_short_name,route_type
4,9031001007500531,14010000721404910,,59.323978,18.094273,1789224113,9011001008200000,9011001008200000,82,1000
31,9031001008000636,14010000684717972,,59.357021,18.110233,1789224106,9011001008000000,9011001008000000,80,1000
67,9031001000500502,14010000725964743,,59.375134,18.336308,1789224110,9011114001100000,9011114001100000,11,1000
88,9031001007500664,14010000717864646,,59.327053,18.076389,1789224107,9011001008400000,9011001008400000,84,1000
97,9031001008000750,14010000684717304,,59.318413,18.158777,1789224104,9011001008000000,9011001008000000,80,1000
157,9031001000500547,14010000729757553,,59.338928,18.209742,1789224097,9011114001100000,9011114001100000,11,1000
159,9031001008000635,14010000684717934,,59.319260,18.098783,1789224112,9011001008000000,9011001008000000,80,1000
163,9031001008000623,14010000694931097,,59.390736,18.109610,1789224109,9011001008000000,9011001008000000,80,1000
171,9031001000500538,14010000729755751,,59.412308,18.359150,1789224107,9011114000900000,9011114000900000,9,1000
211,9031001008000700,14010100726519290,,59.301731,17.876842,1789224104,9011001008900000,9011001008900000,89,1000


In [15]:
# GTFS Sweden 3 är SSOT
# Hämta route-information via:
# realtime vehicle → trip_id → trips.txt → route_id → routes.txt

boats_gtfs = live_trips.merge(
    routes[
        [
            "route_id",
            "route_short_name",
            "route_type"
        ]
    ],
    left_on="route_id_static",
    right_on="route_id",
    how="left"
)

# Endast water transport enligt GTFS Sweden 3
boats_gtfs = boats_gtfs[
    boats_gtfs["route_type"] == "1000"
].copy()

# Behåll bara information från GTFS
boats_gtfs = boats_gtfs[
    [
        "vehicle_id",
        "trip_id",
        "route_id_static",
        "route_short_name",
        "route_type",
        "lat",
        "lon",
        "timestamp"
    ]
].copy()

print("Aktuella båtar från GTFS Sweden 3:", len(boats_gtfs))

display(
    boats_gtfs
    .sort_values(["route_short_name", "vehicle_id"])
    .reset_index(drop=True)
)

Aktuella båtar från GTFS Sweden 3: 37


,vehicle_id,trip_id,route_id_static,route_short_name,route_type,lat,lon,timestamp
0,9031001000500502,14010000725964743,9011114001100000,11,1000,59.375134,18.336308,1789224110
1,9031001000500547,14010000729757553,9011114001100000,11,1000,59.338928,18.209742,1789224097
2,9031001000500536,14010000729759138,9011114001200000,12,1000,59.462147,18.631741,1789224111
3,9031001000500679,14010000733933240,9011114001200000,12,1000,59.457111,18.649292,1789224104
4,9031001000500537,14010000729760454,9011114001300000,13,1000,59.416988,18.353119,1789224109
5,9031001000500539,14010000729761729,9011114001300000,13,1000,59.415150,18.503790,1789224111
6,9031001000500503,14010000725967704,9011114001400000,14,1000,59.401001,18.572548,1789224108
7,9031001000500535,14010000732572657,9011114001400000,14,1000,59.378780,18.871216,1789224108
8,9031001000500540,14010000719263554,9011114001600000,16,1000,59.298382,18.765450,1789224111
9,9031001000500548,14010000729766531,9011114001600000,16,1000,59.303001,18.781807,1789224108


In [16]:
VESSELS_URL = (
    "https://map.stockholmarchipelagotrail.com/data/vessels-identity.json"
)

r_vessels = requests.get(VESSELS_URL)
r_vessels.raise_for_status()

vessels_data = r_vessels.json()

print(type(vessels_data))

<class 'dict'>


In [17]:
# Visa de första posterna i SAT-registret

for key, value in list(vessels_data.items())[:5]:
    print("\nKEY:", key)
    print("VALUE:", value)


KEY: 224xr
VALUE: {'uid': '224xr', 'satId': 'sat:vessel:224xr', 'vehicleId': '9031001000500549', 'name': 'M/S Gällnö', 'wikidata': 'Q15661350', 'concordances': {'gtfs_vehicle': ['gtfs:vehicle:9031001000500549'], 'wikidata': ['wikidata:Q15661350']}, 'status': 'current', 'firstSeen': '2026-07-22'}

KEY: 2r775
VALUE: {'uid': '2r775', 'satId': 'sat:vessel:2r775', 'vehicleId': '9031001000500681', 'name': 'M/S Mysing', 'wikidata': 'Q56839314', 'concordances': {'gtfs_vehicle': ['gtfs:vehicle:9031001000500681'], 'wikidata': ['wikidata:Q56839314']}, 'status': 'current', 'firstSeen': '2026-08-27'}

KEY: 3kjzv
VALUE: {'uid': '3kjzv', 'satId': 'sat:vessel:3kjzv', 'vehicleId': '9031001000500519', 'name': 'M/S Skärgården', 'wikidata': 'Q10573162', 'concordances': {'gtfs_vehicle': ['gtfs:vehicle:9031001000500519'], 'wikidata': ['wikidata:Q10573162']}, 'status': 'current', 'firstSeen': '2026-06-29'}

KEY: 4agqh
VALUE: {'uid': '4agqh', 'satId': 'sat:vessel:4agqh', 'vehicleId': '9031001000500547', 'nam

In [18]:
# Gör SAT-registret till DataFrame
sat_vessels = pd.DataFrame(list(vessels_data.values()))

# Matcha GTFS → SAT
boats_enriched = boats_gtfs.merge(
    sat_vessels[
        [
            "vehicleId",
            "satId",
            "name",
            "wikidata",
            "status"
        ]
    ],
    left_on="vehicle_id",
    right_on="vehicleId",
    how="left"
)

# Markera om GTFS-båten finns i SAT
boats_enriched["sat_match"] = boats_enriched["satId"].notna()

print("GTFS-båtar:", len(boats_enriched))
print("Finns i SAT:", boats_enriched["sat_match"].sum())
print("Saknas i SAT:", (~boats_enriched["sat_match"]).sum())

display(
    boats_enriched[
        [
            "vehicle_id",
            "route_short_name",
            "lat",
            "lon",
            "satId",
            "name",
            "wikidata",
            "sat_match"
        ]
    ].sort_values("route_short_name")
)

GTFS-båtar: 37
Finns i SAT: 19
Saknas i SAT: 18


,vehicle_id,route_short_name,lat,lon,satId,name,wikidata,sat_match
2,9031001000500502,11,59.375134,18.336308,sat:vessel:7bj23,S/S Norrskär,Q10659318,True
5,9031001000500547,11,59.338928,18.209742,sat:vessel:4agqh,Dalarö,Q10573052,True
18,9031001000500679,12,59.457111,18.649292,sat:vessel:zzm4n,M/S Östan,Q64362244,True
35,9031001000500536,12,59.462147,18.631741,sat:vessel:6bv2a,M/S Vånö,Q10573204,True
20,9031001000500539,13,59.415150,18.503790,NaN,NaN,NaN,False
14,9031001000500537,13,59.416988,18.353119,sat:vessel:jr7g7,M/S Väddö,Q10573201,True
31,9031001000500503,14,59.401001,18.572548,sat:vessel:tsvh9,S/S Storskär,Q10659340,True
17,9031001000500535,14,59.378780,18.871216,sat:vessel:af22w,M/S Värmdö,Q10573202,True
10,9031001000500548,16,59.303001,18.781807,sat:vessel:vn92p,M/S Nämdö,Q10573127,True
36,9031001000500540,16,59.298382,18.765450,sat:vessel:5xrhm,M/S Saxaren,Q10573151,True


In [19]:
missing_sat = boats_enriched[
    ~boats_enriched["sat_match"]
].copy()

print("Saknas i SAT:", len(missing_sat))

display(
    missing_sat[
        [
            "vehicle_id",
            "trip_id",
            "route_id_static",
            "route_short_name",
            "lat",
            "lon"
        ]
    ].sort_values("route_short_name")
)

Saknas i SAT: 18


,vehicle_id,trip_id,route_id_static,route_short_name,lat,lon
20,9031001000500539,14010000729761729,9011114001300000,13,59.415150,18.503790
12,9031001000500672,14010000725969702,9011114002600000,26,59.468761,18.479567
15,9031001008000630,14010000684717321,9011001008000000,80,59.319172,18.095316
1,9031001008000636,14010000684717972,9011001008000000,80,59.357021,18.110233
4,9031001008000750,14010000684717304,9011001008000000,80,59.318413,18.158777
6,9031001008000635,14010000684717934,9011001008000000,80,59.319260,18.098783
7,9031001008000623,14010000694931097,9011001008000000,80,59.390736,18.109610
24,9031001008000628,14010000660073444,9011001008000000,80,59.318928,18.144888
22,9031001008000629,14010000660073458,9011001008000000,80,59.317398,18.123554
16,9031001008000634,14010000684717953,9011001008000000,80,59.318466,18.159069


In [20]:
from SPARQLWrapper import SPARQLWrapper, JSON

vehicle_ids = boats_gtfs["vehicle_id"].dropna().unique()

values = " ".join(
    f'"{v}"' for v in vehicle_ids
)

query = f"""
SELECT ?item ?itemLabel ?vehicleId ?MMSIid ?SATid WHERE {{
  VALUES ?vehicleId {{ {values} }}

  ?item wdt:P14632 ?vehicleId .

  OPTIONAL {{ ?item wdt:P587 ?MMSIid }}
  OPTIONAL {{ ?item wdt:P14545 ?SATid }}

  SERVICE wikibase:label {{
    bd:serviceParam wikibase:language "sv,en,mul".
  }}
}}
"""

sparql = SPARQLWrapper(
    "https://query.wikidata.org/sparql"
)

sparql.setQuery(query)
sparql.setReturnFormat(JSON)

# Identifiera oss mot Wikidata
sparql.addCustomHttpHeader(
    "User-Agent",
    "Stockholm Archipelago Trail / salgo60 (salgo60@msn.com)"
)

results = sparql.query().convert()

wikidata_rows = []

for result in results["results"]["bindings"]:
    wikidata_rows.append({
        "wikidata": result["item"]["value"].split("/")[-1],
        "name": result["itemLabel"]["value"],
        "vehicle_id": result["vehicleId"]["value"],
        "MMSI": result.get("MMSIid", {}).get("value"),
        "SATid": result.get("SATid", {}).get("value"),
    })

wikidata = pd.DataFrame(wikidata_rows)

print("Wikidata-matchningar:", len(wikidata))

display(wikidata)

Wikidata-matchningar: 20


,wikidata,name,vehicle_id,MMSI,SATid
0,Q10573052,Dalarö,9031001000500547,265547840,sat:vessel:4agqh
1,Q10573127,M/S Nämdö,9031001000500548,265649360,sat:vessel:vn92p
2,Q10573144,M/S Roslagen,9031001000500520,265522490,sat:vessel:wmf8m
3,Q10573151,M/S Saxaren,9031001000500540,265512110,sat:vessel:5xrhm
4,Q10573158,M/S Sjöbris,9031001000500672,265522700,sat:vessel:y78gd
5,Q10573162,M/S Skärgården,9031001000500519,265522480,sat:vessel:3kjzv
6,Q10573196,M/S Viberö,9031001000500538,265520410,sat:vessel:5vwdh
7,Q10573201,M/S Väddö,9031001000500537,265520420,sat:vessel:jr7g7
8,Q10573202,M/S Värmdö,9031001000500535,265520390,sat:vessel:af22w
9,Q10573203,M/S Västan,9031001000500504,265522440,sat:vessel:7awvt


In [21]:
# Bygg slutlig tabell från GTFS som SSOT
# Alla GTFS-båtar behålls.

boats_final = boats_gtfs.merge(
    wikidata[
        ["vehicle_id", "wikidata", "name", "MMSI", "SATid"]
    ],
    on="vehicle_id",
    how="left"
)

# Skapa matchflagga EFTER merge
boats_final["wikidata_match"] = (
    boats_final["wikidata"].notna()
)

print("GTFS-båtar:", len(boats_final))
print("Wikidata-match:", boats_final["wikidata_match"].sum())
print("Saknas Wikidata:", (~boats_final["wikidata_match"]).sum())




GTFS-båtar: 37
Wikidata-match: 20
Saknas Wikidata: 17


In [22]:
display(
    boats_final[
        [
            "vehicle_id",
            "trip_id",
            "route_id_static",
            "route_short_name",
            "route_type",
            "wikidata",
            "name",
            "MMSI",
            "SATid",
            "lat",
            "lon",
            "wikidata_match"
        ]
    ].sort_values(
        ["wikidata_match", "route_short_name"],
        ascending=[False, True]
    )
)

,vehicle_id,trip_id,route_id_static,route_short_name,route_type,wikidata,name,MMSI,SATid,lat,lon,wikidata_match
2,9031001000500502,14010000725964743,9011114001100000,11,1000,Q10659318,S/S Norrskär,265522430,sat:vessel:7bj23,59.375134,18.336308,True
5,9031001000500547,14010000729757553,9011114001100000,11,1000,Q10573052,Dalarö,265547840,sat:vessel:4agqh,59.338928,18.209742,True
18,9031001000500679,14010000733933240,9011114001200000,12,1000,Q64362244,M/S Östan,265575480,sat:vessel:zzm4n,59.457111,18.649292,True
35,9031001000500536,14010000729759138,9011114001200000,12,1000,Q10573204,M/S Vånö,265520400,sat:vessel:6bv2a,59.462147,18.631741,True
14,9031001000500537,14010000729760454,9011114001300000,13,1000,Q10573201,M/S Väddö,265520420,sat:vessel:jr7g7,59.416988,18.353119,True
17,9031001000500535,14010000732572657,9011114001400000,14,1000,Q10573202,M/S Värmdö,265520390,sat:vessel:af22w,59.378780,18.871216,True
31,9031001000500503,14010000725967704,9011114001400000,14,1000,Q10659340,S/S Storskär,265522420,sat:vessel:tsvh9,59.401001,18.572548,True
10,9031001000500548,14010000729766531,9011114001600000,16,1000,Q10573127,M/S Nämdö,265649360,sat:vessel:vn92p,59.303001,18.781807,True
36,9031001000500540,14010000719263554,9011114001600000,16,1000,Q10573151,M/S Saxaren,265512110,sat:vessel:5xrhm,59.298382,18.765450,True
32,9031001000500520,14010000733559729,9011114001700000,17,1000,Q10573144,M/S Roslagen,265522490,sat:vessel:wmf8m,59.199665,18.429937,True


In [29]:
import folium
import pandas as pd

# --------------------------------------------------
# Lägg till SAT-status från SAT concordance
# --------------------------------------------------

boats_map = boats_final.merge(
    sat_vessels[
        ["vehicleId", "satId"]
    ],
    left_on="vehicle_id",
    right_on="vehicleId",
    how="left"
)

boats_map["sat_match"] = boats_map["satId"].notna()

print("GTFS-båtar:", len(boats_map))
print("SAT finns:", boats_map["sat_match"].sum())
print("SAT saknas:", (~boats_map["sat_match"]).sum())
print("Wikidata finns:", boats_map["wikidata_match"].sum())
print("Wikidata saknas:", (~boats_map["wikidata_match"]).sum())


# --------------------------------------------------
# Karta
# --------------------------------------------------

m = folium.Map(
    location=[59.35, 18.5],
    zoom_start=10,
    tiles="OpenStreetMap"
)


# Fyra ömsesidigt uteslutande lager

sat_wd_group = folium.FeatureGroup(
    name="SAT finns · Wikidata finns"
)

sat_no_wd_group = folium.FeatureGroup(
    name="SAT finns · Wikidata saknas"
)

no_sat_wd_group = folium.FeatureGroup(
    name="SAT saknas · Wikidata finns"
)

no_sat_no_wd_group = folium.FeatureGroup(
    name="SAT saknas · Wikidata saknas"
)


# --------------------------------------------------
# Hösttidtabeller
# --------------------------------------------------

timetable_urls = {
    "2": "https://kund.printhuset-sthlm.se/wa/h2.pdf",
    "3": "https://kund.printhuset-sthlm.se/wa/h3.pdf",
    "4": "https://kund.printhuset-sthlm.se/wa/h4.pdf",
    "5": "https://kund.printhuset-sthlm.se/wa/h5.pdf",
    "7": "https://kund.printhuset-sthlm.se/wa/h7.pdf",
    "8": "https://kund.printhuset-sthlm.se/wa/h8.pdf",
    "9": "https://kund.printhuset-sthlm.se/wa/h9.pdf",
    "11": "https://kund.printhuset-sthlm.se/wa/h11.pdf",
    "12": "https://kund.printhuset-sthlm.se/wa/h12.pdf",
    "13": "https://kund.printhuset-sthlm.se/wa/h13.pdf",
    "14": "https://kund.printhuset-sthlm.se/wa/h14.pdf",
    "15": "https://kund.printhuset-sthlm.se/wa/h15.pdf",
    "26": "https://kund.printhuset-sthlm.se/wa/h26.pdf",
    "27": "https://kund.printhuset-sthlm.se/wa/h27.pdf",
}
23457891112131415171819242627


# --------------------------------------------------
# Lägg ut båtar
# --------------------------------------------------

for _, row in boats_map.iterrows():

    sat_exists = row["sat_match"]
    wikidata_exists = row["wikidata_match"]


    # Välj rätt lager
    if sat_exists and wikidata_exists:
        group = sat_wd_group
        title = row["name"]

    elif sat_exists and not wikidata_exists:
        group = sat_no_wd_group
        title = "SAT · Wikidata saknas"

    elif not sat_exists and wikidata_exists:
        group = no_sat_wd_group
        title = row["name"]

    else:
        group = no_sat_no_wd_group
        title = "SAT saknas · Wikidata saknas"


    # --------------------------------------------------
    # Route / tidtabell
    # --------------------------------------------------

    route = str(row["route_short_name"])

    if route in timetable_urls:
        timetable_link = f"""
        <br>
        <a href="{timetable_urls[route]}" target="_blank">
            📅 Hösttidtabell linje {route}
        </a>
        """
    else:
        timetable_link = ""


    # --------------------------------------------------
    # Popup
    # --------------------------------------------------

    popup = f"""
    <div style="font-family: Arial, sans-serif;">

    <h4 style="margin-bottom: 8px;">
    GTFS / Samtrafiken
    </h4>

    <b>Vehicle ID</b><br>
    <span style="font-family: monospace;">
    {row['vehicle_id']}
    </span><br><br>

    <b>Trip ID</b><br>
    <span style="font-family: monospace;">
    {row['trip_id']}
    </span><br><br>

    <b>Route</b><br>
    Linje {route}
    {timetable_link}<br><br>

    <b>Route type</b><br>
    {row['route_type']}<br><br>

    <b>SAT</b><br>
    {"✓ Finns" if sat_exists else "✗ Saknas"}<br>

    <b>Wikidata</b><br>
    {"✓ Finns" if wikidata_exists else "✗ Saknas"}<br><br>

    <b>Position</b><br>
    Lat: {row['lat']}<br>
    Lon: {row['lon']}<br>
    """


    # --------------------------------------------------
    # Wikidata
    # --------------------------------------------------

    if wikidata_exists:

        popup += f"""
        <hr>

        <a href="https://www.wikidata.org/wiki/{row['wikidata']}"
           target="_blank">
           🔗 Wikidata – {row['name']}
        </a>
        """

        if pd.notna(row["MMSI"]):

            popup += f"""
            <br>
            <a href="https://www.vesselfinder.com/vessels/details/{row['MMSI']}"
               target="_blank">
               🚢 VesselFinder – MMSI {row['MMSI']}
            </a>
            """

        if pd.notna(row["SATid"]):

            sat_url = (
                "https://map.stockholmarchipelagotrail.com/"
                "?id=" + str(row["SATid"]).replace(":", "%3A")
            )

            popup += f"""
            <br>
            <a href="{sat_url}" target="_blank">
               🗺 SAT – {row['SATid']}
            </a>
            """


    popup += """
    </div>
    """


    # --------------------------------------------------
    # Marker
    # --------------------------------------------------

    folium.Marker(
        location=[row["lat"], row["lon"]],
        tooltip=title,
        popup=folium.Popup(
            popup,
            max_width=500
        )
    ).add_to(group)


# --------------------------------------------------
# Lägg lager på kartan
# --------------------------------------------------

sat_wd_group.add_to(m)
sat_no_wd_group.add_to(m)
no_sat_wd_group.add_to(m)
no_sat_no_wd_group.add_to(m)

folium.LayerControl(
    collapsed=False
).add_to(m)

m

GTFS-båtar: 37
SAT finns: 19
SAT saknas: 18
Wikidata finns: 20
Wikidata saknas: 17
